# AI Tools for Research: Citation Metadata Consistency Pilot

**Self-contained Google Colab experiment (CPU only).** This notebook includes its own reference catalog, program and frozen protocol. Upload this `.ipynb` to Colab and select **Runtime → Run all**. No external dataset, API key or GPU is required.

**Scientific distinction:** this evaluates whether a citation's fields agree with an *offline reviewed reference inventory*. It does **not** detect trust in readers, establish whether a claim is supported by an article, or measure real LLM hallucination rates. Five hundred? **Exactly 650** synthetic cases are constructed by fixed rules from 13 bibliographic source records. Three of those records were independently spot-checked; ten remain review-derived and need primary-source verification.

**Expected locally measured holdout F1:** strict 0.333, title-only 0.667, metadata-aware 1.000. A **separate exploratory stress test** shows why 1.000 is not a universal accuracy claim: 5 of 13 altered titles were falsely accepted by the metadata-aware method.

Run every cell sequentially; last cell creates a ZIP of the actual notebook outputs.

In [ ]:
import sys, importlib.util, subprocess, platform
required = {'numpy':'numpy>=1.24,<3', 'pandas':'pandas>=2.0,<3', 'matplotlib':'matplotlib>=3.6,<4', 'sklearn':'scikit-learn>=1.2,<2'}
missing = [spec for mod,spec in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:',missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('Required numerical libraries are already present.')
print('Python:',sys.version.split()[0], 'OS:',platform.platform())


In [ ]:
from pathlib import Path
import os, json
IN_COLAB = 'google.colab' in sys.modules
ROOT = Path('/content/CitationAudit_Colab_Output') if IN_COLAB else Path.cwd()/'CitationAudit_Colab_Output'
(ROOT/'data').mkdir(parents=True,exist_ok=True)
(ROOT/'configs').mkdir(parents=True,exist_ok=True)
(ROOT/'results').mkdir(parents=True,exist_ok=True)
seed_csv = 'ref_id,title,first_author,year,doi,metadata_origin,independent_status\nR01,Artificial hallucinations in ChatGPT: Implications in scientific writing,Alkaissi,2023,10.7759/cureus.35179,review_reference_1,review_transcribed\nR02,Exploring the boundaries of reality: Investigating the phenomenon of artificial intelligence hallucination in scientific writing through ChatGPT references,Athaluri,2023,10.7759/cureus.37432,review_reference_2,review_transcribed\nR03,Does the whole exceed its parts? The effect of AI explanations on complementary team performance,Bansal,2021,10.1145/3411764.3445717,review_reference_3,review_transcribed\nR04,High rates of fabricated and inaccurate references in ChatGPT-generated medical content,Bhattacharyya,2023,10.7759/cureus.39238,review_reference_4,publisher_and_pubmed_spotchecked\nR05,To trust or to think: Cognitive forcing functions can reduce overreliance on AI in AI-assisted decision-making,Bucinca,2021,10.1145/3449287,review_reference_5,review_transcribed\nR06,A preliminary investigation of fake peer-reviewed citations and references generated by ChatGPT,Day,2023,10.1080/00330124.2023.2190373,review_reference_6,review_transcribed\nR07,Fostering appropriate reliance on large language models: The role of explanations sources and inconsistencies,Kim,2025,10.1145/3706598.3714020,review_reference_7,review_transcribed\nR08,Perceived credibility of Internet encyclopedias,Kubiszewski,2011,10.1016/j.compedu.2010.10.008,review_reference_8,review_transcribed\nR09,Nonprobative photographs (or words) inflate truthiness,Newman,2012,10.3758/s13423-012-0292-0,review_reference_10,review_transcribed\nR10,Truthiness and falsiness of trivia claims depend on judgmental contexts,Newman,2015,10.1037/xlm0000099,review_reference_11,review_transcribed\nR11,Prior exposure increases perceived accuracy of fake news,Pennycook,2018,10.1037/xge0000465,review_reference_12,review_transcribed\nR12,The citation effect: In-text citations moderately increase belief in trivia claims,Putnam,2017,10.1016/j.actpsy.2017.07.010,review_reference_14,publisher_and_pubmed_spotchecked\nR13,Fabrication and errors in the bibliographic citations generated by ChatGPT,Walters,2023,10.1038/s41598-023-41032-5,review_reference_15,publisher_and_pubmed_spotchecked\n'
protocol_json = '{\n  "protocol_status": "frozen_before_the_first_project_run",\n  "review_topic": "Citation formatting, fabricated references and human trust",\n  "operational_task": "Internal metadata consistency with a frozen offline reference catalog (not human trust)",\n  "design": "Controlled computational comparison, review-derived DOI metadata plus controlled synthetic perturbations",\n  "dataset": "13 manually transcribed DOI-bearing references from the user\'s prior review",\n  "positive_variations": ["as_written", "upper_case", "punctuation", "doi_url", "minor_title_typo"],\n  "negative_variations": ["wrong_doi", "wrong_title", "wrong_author", "wrong_year", "nonexistent_doi"],\n  "methods": ["strict_exact", "title_fuzzy", "metadata_aware"],\n  "ablation_methods": ["metadata_no_doi", "metadata_no_author", "metadata_no_year", "metadata_no_title"],\n  "frozen_source_split_seed": 20261117,\n  "perturbation_seeds": [11, 23, 47, 59, 83],\n  "ratio_holdout_sources_approx": 0.4,\n  "instances_per_source_per_seed": 10,\n  "primary_metric": "F1 (holdout source records; macro-average over five seeds)",\n  "secondary_metrics": ["precision", "recall", "balanced_accuracy", "false_acceptance_rate", "false_rejection_rate"],\n  "title_fuzzy_threshold": 0.86,\n  "metadata_title_similarity_threshold": 0.89,\n  "higher_is_better": ["F1", "precision", "recall", "balanced_accuracy"],\n  "lower_is_better": ["false_acceptance_rate", "false_rejection_rate"],\n  "planned_deeper_analyses": ["error_type_accuracy", "metadata_field_ablation"],\n  "hypotheses": ["Metadata-aware matching reduces false accepts of inconsistent references", "Exact matching is brittle to benign formatting changes"],\n  "primary_decision_rule": "compare holdout F1 and false acceptance, avoiding any post-hoc tuning",\n  "critical_limitations": ["13-source small catalog", "synthetic corruptions", "no observed AI-generation or real human trust data", "offline DOI lookup only", "catalog contains known references for both partitions"]\n}\n'
(ROOT/'data'/'seed_references.csv').write_text(seed_csv,encoding='utf-8')
(ROOT/'configs'/'frozen_protocol.json').write_text(protocol_json,encoding='utf-8')
print('Run directory:',ROOT)
print('Source records in embedded CSV:',len(seed_csv.strip().splitlines())-1)


## Algorithm implementation

Three comparisons: **strict exact**, **title-only fuzzy**, **metadata-aware DOI→title→first-author→year checks**, plus four field ablations. Below is the same source code used to generate the packaged local CSV and figure outputs. Class labels refer only to intentionally constructed metadata mismatches.

In [ ]:
"""Citation consistency experiment. Input is curated bibliographic records.

This project tests bibliographic METADATA CONSISTENCY, not whether a source
supports a scientific claim and not human trust in a citation.
All perturbations are generated deliberately; none is an observed LLM error.
"""
from __future__ import annotations
import csv
import json
import platform
import re
import sys
import time
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path
from collections import defaultdict
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEEDS = [11, 23, 47, 59, 83]
SPLIT_SEED = 20261117
METHODS = ['strict_exact', 'title_fuzzy', 'metadata_aware']
ABLATIONS = ['metadata_no_doi', 'metadata_no_author', 'metadata_no_year', 'metadata_no_title']
POS_VARIANTS = ['as_written', 'upper_case', 'punctuation', 'doi_url', 'minor_title_typo']
NEG_VARIANTS = ['wrong_doi', 'wrong_title', 'wrong_author', 'wrong_year', 'nonexistent_doi']
TITLE_THRESHOLD = 0.86
STRICT_TITLE_THRESHOLD = 0.89


def norm(value):
    return ' '.join(re.findall(r'[a-z0-9]+', str(value).casefold()))


def doi_norm(doi):
    value = str(doi).strip().lower()
    value = re.sub(r'^https?://(?:dx\.)?doi\.org/', '', value)
    value = re.sub(r'^doi\s*:\s*', '', value)
    return value.rstrip(' .;,')


def title_similarity(a, b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()


def load_records(path):
    d = pd.read_csv(path, dtype={'year': int, 'doi': str})
    required = {'ref_id', 'title', 'first_author', 'year', 'doi'}
    assert required <= set(d.columns)
    assert d.ref_id.is_unique and d.doi.is_unique
    assert d[sorted(required)].notna().all().all()
    assert (d['year'].between(1990, 2026)).all()
    return d.to_dict('records')


def split_sources(records):
    ids = sorted(r['ref_id'] for r in records)
    random.Random(SPLIT_SEED).shuffle(ids)
    n_hold = max(3, round(len(ids) * 0.4))
    return set(ids[n_hold:]), set(ids[:n_hold])


def alter_one_letter(title, rng):
    # A predictable, mild transcription variation, preserving semantics.
    candidates = [i for i, c in enumerate(title) if c.lower() in 'abcdefghijklmnopqrstuvwxyz' and i > 5]
    i = rng.choice(candidates)
    replacement = 'x' if title[i].lower() != 'x' else 'z'
    return title[:i] + replacement + title[i+1:]


def make_example(base, variation, others, rng, seed):
    # All records are copied before perturbation, preventing source mutation.
    record = {k: base[k] for k in ['ref_id', 'title', 'first_author', 'year', 'doi']}
    record.update({'variation': variation, 'generator_seed': seed, 'source_id': base['ref_id']})
    if variation in POS_VARIANTS:
        record['is_consistent'] = 1
        if variation == 'upper_case':
            record['title'] = base['title'].upper()
            record['first_author'] = base['first_author'].upper()
        elif variation == 'punctuation':
            record['title'] = '  ' + re.sub(r'[:?—-]', ' ', base['title']) + ' !! '
        elif variation == 'doi_url':
            record['doi'] = 'https://doi.org/' + base['doi']
        elif variation == 'minor_title_typo':
            record['title'] = alter_one_letter(base['title'], rng)
    elif variation in NEG_VARIANTS:
        record['is_consistent'] = 0
        other = rng.choice([x for x in others if x['ref_id'] != base['ref_id']])
        if variation == 'wrong_doi':
            record['doi'] = other['doi']
        elif variation == 'wrong_title':
            record['title'] = other['title']
        elif variation == 'wrong_author':
            candidate = [x for x in others if x['first_author'].lower() != base['first_author'].lower()]
            record['first_author'] = rng.choice(candidate)['first_author']
        elif variation == 'wrong_year':
            record['year'] = int(base['year']) + rng.choice([-4, -3, 3, 4])
        elif variation == 'nonexistent_doi':
            # Deliberately synthetic ID; NOT asserted to be registry-checked/nonexistent.
            record['doi'] = '10.0000/experiment-' + str(seed) + '-' + base['ref_id']
    else:
        raise ValueError(variation)
    record['citation_text'] = (f"{record['first_author']} ({record['year']}). "
        f"{record['title']}. doi:{record['doi']}")
    record['sample_id'] = f"s{seed}:{base['ref_id']}:{variation}"
    return record


def build_dataset(records, seeds=SEEDS):
    out = []
    for seed in seeds:
        rng = random.Random(seed)
        for record in records:
            for variation in POS_VARIANTS + NEG_VARIANTS:
                out.append(make_example(record, variation, records, rng, seed))
    df = pd.DataFrame(out)
    dev_ids, holdout_ids = split_sources(records)
    df['partition'] = df.source_id.map(lambda x: 'holdout' if x in holdout_ids else 'development')
    assert df.sample_id.is_unique
    assert df.groupby(['source_id', 'generator_seed']).size().eq(10).all()
    assert all(df[df.partition == x].is_consistent.mean() == .5 for x in ['holdout', 'development'])
    assert not set(df[df.partition == 'holdout'].source_id) & set(df[df.partition == 'development'].source_id)
    return df


def candidate_record(sample, catalog):
    # Real registry use would retrieve an external source. This offline experiment
    # selects the nearest title among its frozen reviewed bibliographic catalog.
    return max(catalog, key=lambda r: title_similarity(sample['title'], r['title']))


def predict(sample, catalog, method):
    if method == 'strict_exact':
        return int(any(
            str(sample['title']) == r['title'] and
            str(sample['first_author']) == r['first_author'] and
            int(sample['year']) == int(r['year']) and
            str(sample['doi']) == r['doi'] for r in catalog))
    nearest = candidate_record(sample, catalog)
    similarity = title_similarity(sample['title'], nearest['title'])
    if method == 'title_fuzzy':
        return int(similarity >= TITLE_THRESHOLD)
    ignore_doi = method == 'metadata_no_doi'
    ignore_author = method == 'metadata_no_author'
    ignore_year = method == 'metadata_no_year'
    ignore_title = method == 'metadata_no_title'
    if method not in ['metadata_aware'] + ABLATIONS:
        raise ValueError(method)
    # DOI resolution against the frozen catalog is attempted first; absence in
    # this catalog is treated as not found, NOT proof of real-world nonexistence.
    doi = doi_norm(sample['doi'])
    doi_matches = [r for r in catalog if doi_norm(r['doi']) == doi]
    if doi_matches and not ignore_doi:
        nearest = doi_matches[0]
        similarity = title_similarity(sample['title'], nearest['title'])
    elif not ignore_doi:
        return 0
    # When DOI checks are ablated, title is the lookup key.
    return int(
        (ignore_title or similarity >= STRICT_TITLE_THRESHOLD) and
        (ignore_author or norm(sample['first_author']) == norm(nearest['first_author'])) and
        (ignore_year or int(sample['year']) == int(nearest['year'])) and
        (ignore_doi or doi_norm(sample['doi']) == doi_norm(nearest['doi']))
    )


def evaluate_predictions(df, methods):
    results = []
    for (method, seed, partition), group in df.groupby(['method', 'generator_seed', 'partition']):
        if method not in methods: continue
        y = group.is_consistent.to_numpy()
        p = group.prediction.to_numpy()
        tn, fp, fn, tp = confusion_matrix(y, p, labels=[0,1]).ravel()
        results.append(dict(method=method, generator_seed=int(seed), partition=partition, n=len(y),
            accuracy=accuracy_score(y,p), balanced_accuracy=balanced_accuracy_score(y,p),
            precision=precision_score(y,p,zero_division=0), recall=recall_score(y,p,zero_division=0),
            f1=f1_score(y,p,zero_division=0), false_acceptance_rate=fp/(fp+tn),
            false_rejection_rate=fn/(fn+tp), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp)))
    return pd.DataFrame(results).sort_values(['partition', 'method', 'generator_seed'])


def save_plot(fig, stem, out):
    fig.savefig(out / f'{stem}.png', dpi=180, bbox_inches='tight')
    fig.savefig(out / f'{stem}.pdf', bbox_inches='tight')
    plt.close(fig)


def make_figures(metrics, predictions, figs):
    figs.mkdir(parents=True,exist_ok=True)
    sub = metrics[(metrics.partition == 'holdout') & metrics.method.isin(METHODS)]
    dat = sub.groupby('method')[['f1','false_acceptance_rate','recall']].mean().reindex(METHODS)
    fig, ax = plt.subplots(figsize=(7.4,4.2))
    xx = np.arange(len(dat)); w = .25
    for j, col in enumerate(dat.columns):
        ax.bar(xx+(j-1)*w, dat[col].values,w,label=col.replace('_',' ').title())
    ax.set_xticks(xx, ['Strict exact','Title fuzzy','Metadata aware']); ax.set_ylim(0,1.04)
    ax.set_ylabel('Fraction (higher is better except false-acceptance rate)')
    ax.legend(loc='upper left',fontsize=8); ax.grid(axis='y',alpha=.22)
    fig.tight_layout(); save_plot(fig,'methods_comparison',figs)
    subp = predictions[(predictions.partition=='holdout') & (predictions.method.isin(METHODS))]
    gp = subp.groupby(['method','variation']).apply(
        lambda q: (q.prediction == q.is_consistent).mean(),include_groups=False).unstack().reindex(METHODS)
    gp.to_csv(figs.parent / 'variation_accuracy.csv')
    fig, ax = plt.subplots(figsize=(9,4.1))
    im=ax.imshow(gp.values,aspect='auto',vmin=0,vmax=1,cmap='Blues')
    ax.set_xticks(np.arange(len(gp.columns)),gp.columns,rotation=40,ha='right',fontsize=8)
    ax.set_yticks(np.arange(3),['Strict exact','Title fuzzy','Metadata aware'])
    for i in range(gp.shape[0]):
        for j in range(gp.shape[1]):
            v=gp.values[i,j]; ax.text(j,i,f'{v:.2f}',ha='center',va='center',color='white' if v>.65 else 'black',fontsize=7)
    fig.colorbar(im,ax=ax,fraction=.03,label='Accuracy'); fig.tight_layout()
    save_plot(fig,'variation_robustness',figs)
    sub = metrics[(metrics.partition=='holdout') & (metrics.method.isin(['metadata_aware'] + ABLATIONS))]
    agg=sub.groupby('method')['f1'].mean().reindex(['metadata_aware']+ABLATIONS)
    fig,ax=plt.subplots(figsize=(7.3,3.8))
    ax.barh(np.arange(len(agg)),agg.values)
    ax.set_yticks(np.arange(len(agg)),['Full model','Without DOI','Without author','Without year','Without title']); ax.invert_yaxis()
    ax.set_xlabel('Holdout F1'); ax.set_xlim(0,1.02); ax.grid(axis='x',alpha=.2)
    fig.tight_layout();save_plot(fig,'ablation_f1',figs)


def run_project(project_root, run_label='local_run'):
    root=Path(project_root); result=root/'results'; result.mkdir(parents=True,exist_ok=True)
    records=load_records(root/'data'/'seed_references.csv')
    data=build_dataset(records)
    all_methods=METHODS+ABLATIONS
    catalog=records # Frozen known-source catalog is available for verifying BOTH partitions.
    output=[]
    start=time.perf_counter()
    for method in all_methods:
        for row in data.to_dict('records'):
            sample=row.copy();sample['method']=method;sample['prediction']=predict(row,catalog,method)
            output.append(sample)
    elapsed=time.perf_counter()-start
    predictions=pd.DataFrame(output).sort_values(['method','sample_id'])
    metrics=evaluate_predictions(predictions,all_methods)
    predictions.to_csv(result/'predictions.csv',index=False)
    data.to_csv(result/'synthetic_test_cases.csv',index=False)
    metrics.to_csv(result/'metrics_by_seed.csv',index=False)
    summary=(metrics[metrics.partition=='holdout'].groupby('method')
        [['f1','precision','recall','accuracy','balanced_accuracy','false_acceptance_rate','false_rejection_rate']]
        .agg(['mean','std']).round(6))
    summary.columns=['_'.join(x) for x in summary.columns]
    summary.to_csv(result/'results.csv')
    make_figures(metrics,predictions,result/'figures')
    dev_ids,holdout_ids=split_sources(records)
    audit={'run_label':run_label,'source_record_count':len(records),
        'test_cases_count':len(data),'predictions_count':len(predictions),
        'seeds':SEEDS,'split_seed':SPLIT_SEED,
        'development_source_ids':sorted(dev_ids),'holdout_source_ids':sorted(holdout_ids),
        'comparison_methods':all_methods,'elapsed_processing_seconds':round(elapsed,4),
        'python_version':sys.version,'platform':platform.platform(),
        'numpy_version':np.__version__,'pandas_version':pd.__version__,
        'matplotlib_version':matplotlib.__version__,
        'sklearn_version':__import__('sklearn').__version__,
        'verification_scope':'offline comparison against reviewed 13-record catalog; not live DOI lookup or human trust',
        'data_status':'genuine reviewed bibliographic metadata with deliberately synthetic perturbations'}
    (result/'run_manifest.json').write_text(json.dumps(audit,indent=2),encoding='utf-8')
    print('Sources:',len(records),'test cases:',len(data),'predictions:',len(predictions))
    print('Holdout references:', ', '.join(sorted(holdout_ids)))
    print(summary.to_string())
    return summary,audit

In [ ]:
from IPython.display import display
records = load_records(ROOT/'data'/'seed_references.csv')
examples = build_dataset(records)
assert len(records)==13 and len(examples)==650
assert len(set(examples[examples.partition=='holdout'].source_id).intersection(set(examples[examples.partition=='development'].source_id)))==0
assert examples.is_consistent.mean()==.5
print('DATA VALIDATION PASSED:',len(records),'references,',len(examples),'cases')
print('Source partition counts:',examples.groupby('partition').source_id.nunique().to_dict())
print('Local research statements: source catalog checks metadata; no human trust data.')


## Primary frozen experiment

The same 650 cases are given to each of seven rules; only holdout metrics are summarized. All recorded results here come from *this notebook's actual execution*, not placeholders.

In [ ]:
summary, manifest = run_project(ROOT,run_label='colab_notebook_run')
print('Primary metrics, mean across five controlled seeds:')
show = summary[['f1_mean','precision_mean','recall_mean','false_acceptance_rate_mean']].loc[METHODS]
display(show)
assert len(pd.read_csv(ROOT/'results'/'predictions.csv'))==4550


## Exploratory *post-hoc* stress check

Added after the frozen primary benchmark was observed: each original title is wrongly prefixed by `A systematic review of`, leaving its DOI, year and author unchanged. False acceptances count whether a rule incorrectly accepts a now-different title. These thirteen cases **do not alter the primary metric**.

In [ ]:
stress=[]
for r in records:
    sample=dict(r)
    sample['title']='A systematic review of ' + r['title']
    row={'ref_id':r['ref_id'],'original_title':r['title'],'altered_title':sample['title'],
         'title_similarity':title_similarity(sample['title'],r['title']),
         'strict_exact':predict(sample,records,'strict_exact'),
         'title_fuzzy':predict(sample,records,'title_fuzzy'),
         'metadata_aware':predict(sample,records,'metadata_aware'),
         'actual_consistency':0}
    stress.append(row)
stress_df=pd.DataFrame(stress)
stress_df.to_csv(ROOT/'results'/'exploratory_stress_test.csv',index=False)
print('POST-HOC stress false acceptances out of 13:')
print(stress_df[['strict_exact','title_fuzzy','metadata_aware']].sum().to_string())
assert len(stress_df)==13
assert int(stress_df.metadata_aware.sum())==5


## Figures and exported evidence

Each chart is generated from predictions saved during this run, not illustrative dummy values.

In [ ]:
from IPython.display import Image
for stem in ['methods_comparison','variation_robustness','ablation_f1']:
    print('Actual generated figure:',stem)
    display(Image(filename=str(ROOT/'results'/'figures'/f'{stem}.png')))


In [ ]:
import shutil
zip_path = shutil.make_archive(str(ROOT), 'zip', root_dir=ROOT)
print('Actual notebook outputs zipped:',zip_path)
print('Files:',len(list(ROOT.rglob('*'))))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print('Not running inside hosted Colab: use the generated local path above.')


## Interpretation / limitations

* Observed primary holdout F1 values may be **1.0** for the metadata-aware method because the perturbations directly alter precisely the fields checked in a small closed catalog. This is not proof of real-world accuracy.
* The extra post-hoc check shows a real failure in this code on deliberately altered titles (5/13 falsely accepted); it must not be omitted when discussing the result.
* The study does not test trust ratings, participant decisions, claim support, complete online DOI existence, unseen real publications, or realistic natural language model hallucination samples.
* Source-level split groups all synthetic versions of a source, but the lookup catalog intentionally contains the five holdout originals. Thus the 'holdout' measures matching robustness, **not discovery of completely unseen bibliography items**.
* Source provenance: three entries checked against publisher/PubMed; ten remain transcribed from the student's prior review.

For a technical viva, be prepared to explain why exact matching fails on DOI URL formatting, why title-only matching misses wrong authors/years/DOIs, why synthetic perfect F1 is misleading, and why the stress test is labeled post-hoc.